In [1]:
!unzip -q "/content/qwen2.5-3b-roadbuddy-ft.zip" -d "/content"


/content/content/qwen2.5-3b-roadbuddy-ft/checkpoint-26/adapter_model.safetensors  bad CRC bcfb3652  (should be e4835e4f)


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [31]:
# ==================== CELL 0 — CÀI ĐẶT THƯ VIỆN (SETUP) ====================

# 🔴 CHẠY CELL NÀY ĐẦU TIÊN 🔴

print("🔄 Installing dependencies...")

# Core libraries
!pip install -q ultralytics        # YOLOv11
!pip install -q easyocr            # OCR
!pip install -q sentence-transformers  # Embeddings
!pip install -q faiss-cpu          # Vector search (hoặc faiss-gpu nếu có GPU)

# LLM và quantization
!pip install -q "transformers>=4.41.0"
!pip install -q "accelerate>=0.25.0"
!pip install -q bitsandbytes einops tiktoken qwen-vl-utils

# PDF processing
!pip install -q pypdf              # Hoặc PyPDF2

# Image processing
!pip install -q pillow opencv-python imageio

# Deep learning (CUDA)
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

print("✅ Installation complete!")


🔄 Installing dependencies...
✅ Installation complete!


In [32]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="omUZ7qdRSYyo5lYPUVGx")
project = rf.workspace("myplan-qv6ss").project("my-traffic-train-ygas8")
version = project.version(5)
dataset = version.download("yolov11")


loading Roboflow workspace...
loading Roboflow project...


KeyboardInterrupt: 

In [33]:
# ==================== CELL 1 — IMPORT THƯ VIỆN & KIẾN TRÚC CHUNG ====================

import os, re, json, time, csv, gc, math, shutil
from pathlib import Path
from typing import List, Dict, Any, Tuple, Optional
from collections import Counter, defaultdict

import numpy as np
import cv2
from PIL import Image

# YOLOv11
from ultralytics import YOLO
import torch

# OCR + Embeddings + RAG
import easyocr
import faiss
from sentence_transformers import SentenceTransformer

# Transformer-based LLM
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# CLIP (optional)
try:
    from transformers import AutoProcessor, CLIPModel
    _CLIP_AVAILABLE = True
except:
    _CLIP_AVAILABLE = False

# PDF processing
try:
    from pypdf import PdfReader
except:
    try:
        from PyPDF2 import PdfReader
    except:
        PdfReader = None

print("✅ All libraries imported successfully!")


✅ All libraries imported successfully!


In [37]:
# ==================== CELL 2 — CẤU HÌNH ĐƯỜNG DẪN & THAM SỐ ====================
from pathlib import Path
import json
import torch

QUESTIONS_JSON = "/content/public_test.json"
VIDEO_ROOT     = "/content/drive/MyDrive/videos"

# Model gốc
WEIGHTS_BASE = "/content/bestfinal.pt"

# Nếu đã có model fine-tune thì dùng, còn không thì dùng model gốc
FINETUNE_WEIGHTS = "/content/roadbuddy/runs_finetune/yolo11_roadbuddy_finetune/weights/best.pt"
if Path(FINETUNE_WEIGHTS).exists():
    WEIGHTS = FINETUNE_WEIGHTS
else:
    WEIGHTS = WEIGHTS_BASE

# Dataset YOLO ở Roboflow đã tải về
YOLO_DATA_YAML = "/content/My-Traffic-Train-5/data.yaml"

print("✅ YOLO CONFIG LOADED")
print("   WEIGHTS_BASE   :", WEIGHTS_BASE)
print("   WEIGHTS dùng   :", WEIGHTS)
print("   YOLO_DATA_YAML :", YOLO_DATA_YAML)

# PDF Luật (nếu có)
LAW_PDF_NEW = "/content/luatgiaothong.pdf"
LAW_PDF_OLD = "/content/luatbosung.pdf"

# Runtime cache
CACHE_DIR = "roadbuddy/runtime/cache"
KB_DIR = "roadbuddy/kb"
RAG_DIR = f"{KB_DIR}/index"

Path(CACHE_DIR).mkdir(parents=True, exist_ok=True)
Path(RAG_DIR).mkdir(parents=True, exist_ok=True)

# Outputs
OUT_SHORT = "roadbuddy/runtime/answers_short.csv"
OUT_FULL  = "roadbuddy/runtime/answers_full.csv"

# Models
EMB_MODEL   = "sentence-transformers/all-MiniLM-L6-v2"
CLIP_MODEL  = "openai/clip-vit-base-patch32"

QWEN_MODEL   = "Qwen/Qwen2.5-3B-Instruct"
QWEN_ADAPTER = "/content/content/qwen2.5-3b-roadbuddy-ft"   # thư mục chứa LoRA FT

USE_QWEN         = True
USE_CLIP_CONTEXT = True  # Bật nếu muốn dùng CLIP embedding

# Detection params
CONF_THRES = 0.25
IOU_THRES  = 0.45
DEVICE     = 0 if torch.cuda.is_available() else "cpu"

# Keyframe params
KEYFRAME_MODE      = "time"
SECONDS_PER_SAMPLE = 1.0
MAX_FRAMES_PER_VID = 60
SCENE_THRESHOLD    = 0.40

# Scene description params
WIN_SEC      = 2.0
FPS_FALLBACK = 30.0

# Labels
LABEL_FILE_PATH = "/content/label.json"
LABEL_MAP = {}

print("✅ Configuration loaded")

# Load LABEL_MAP
if Path(LABEL_FILE_PATH).exists():
    try:
        with open(LABEL_FILE_PATH, "r", encoding="utf-8") as f:
            LABEL_MAP = json.load(f)
        print(f"✅ Label map loaded from {LABEL_FILE_PATH}")
    except Exception as e:
        print(f"⚠️ Error loading label map from {LABEL_FILE_PATH}: {e}")
else:
    print(f"⚠️ Label file not found: {LABEL_FILE_PATH}. Using default labels.")


✅ YOLO CONFIG LOADED
   WEIGHTS_BASE   : /content/bestfinal.pt
   WEIGHTS dùng   : /content/roadbuddy/runs_finetune/yolo11_roadbuddy_finetune/weights/best.pt
   YOLO_DATA_YAML : /content/My-Traffic-Train-5/data.yaml
✅ Configuration loaded
✅ Label map loaded from /content/label.json


In [35]:
# ==================== CELL 3 — HÀM KHỞI TẠO CÁC MODEL TOÀN CỤC ====================
from peft import PeftModel

_yolo_model = None
_ocr_reader = None
_emb_model = None
_qwen_model = None
_qwen_tok = None
_clip_model = None
_clip_proc = None
_rag_index = None
_rag_chunks = []

def init_yolo(force_reload: bool = False):
    """
    Hàm load YOLO một lần duy nhất.
    Sau khi fine-tune, WEIGHTS sẽ đổi → reset model → load lại.
    """
    global _yolo_model, WEIGHTS

    if force_reload or _yolo_model is None:
        device = 0 if torch.cuda.is_available() else "cpu"
        print(f"🔧 Loading YOLO with weights: {WEIGHTS} (device={device})")
        _yolo_model = YOLO(WEIGHTS)
    return _yolo_model

def init_ocr():
    global _ocr_reader
    if _ocr_reader is None:
        print("🔄 Loading OCR...")
        _ocr_reader = easyocr.Reader(['vi','en'], gpu=torch.cuda.is_available())
    return _ocr_reader

def init_embeddings():
    global _emb_model
    if _emb_model is None:
        print(f"🔄 Loading Embeddings: {EMB_MODEL}")
        _emb_model = SentenceTransformer(EMB_MODEL)
    return _emb_model

def init_clip():
    global _clip_model, _clip_proc
    if USE_CLIP_CONTEXT and _CLIP_AVAILABLE and _clip_model is None:
        print(f"🔄 Loading CLIP: {CLIP_MODEL}")
        _clip_proc = AutoProcessor.from_pretrained(CLIP_MODEL)
        _clip_model = CLIPModel.from_pretrained(CLIP_MODEL).to(DEVICE).eval()
    return _clip_model, _clip_proc



def init_qwen():
    global _qwen_model, _qwen_tok
    if not USE_QWEN or _qwen_model is not None:
        return _qwen_model, _qwen_tok

    print(f"🔄 Loading Qwen base: {QWEN_MODEL}")
    try:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4"
        )
        _qwen_tok = AutoTokenizer.from_pretrained(QWEN_MODEL, trust_remote_code=True)
        base_model = AutoModelForCausalLM.from_pretrained(
            QWEN_MODEL,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True,
            low_cpu_mem_usage=True,
            torch_dtype=torch.float16
        )

        print(f"🔄 Loading QLoRA adapter from: {QWEN_ADAPTER}")
        _qwen_model = PeftModel.from_pretrained(base_model, QWEN_ADAPTER)

        print("✅ Qwen + LoRA (RoadBuddy FT) loaded")
    except Exception as e:
        print(f"⚠️ Qwen load failed: {e}")
        _qwen_model = None

    return _qwen_model, _qwen_tok


print("✅ Model initializers ready")

✅ Model initializers ready


In [38]:
# ==================== CELL X — FINE-TUNE YOLO TRÊN DATASET ====================
from ultralytics import YOLO
from pathlib import Path
import torch

# ⚙️ 1. CẤU HÌNH DATASET & THAM SỐ HUẤN LUYỆN
DATA_YAML = YOLO_DATA_YAML          # dùng chung với config
BASE_WEIGHTS = WEIGHTS_BASE         # 🔴 LUÔN fine-tune từ model gốc

EPOCHS      = 5
BATCH_SIZE  = 16
IMG_SIZE    = 640
PROJECT     = "roadbuddy/runs_finetune"
RUN_NAME    = "yolo11_roadbuddy_finetune"

print("🚀 Bắt đầu fine-tune YOLO...")
print(f"📁 DATA_YAML    : {DATA_YAML}")
print(f"🎯 BASE_WEIGHTS : {BASE_WEIGHTS}")

# ⚙️ 2. KHỞI TẠO MODEL
model = YOLO(BASE_WEIGHTS)

# ⚙️ 3. TRAIN / FINE-TUNE
results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    project=PROJECT,
    name=RUN_NAME,
    exist_ok=True,
    patience=10,
    device=0 if torch.cuda.is_available() else "cpu"
)

# ⚙️ 4. CẬP NHẬT WEIGHTS SAU TRAIN
save_dir = None
if hasattr(model, "trainer") and hasattr(model.trainer, "save_dir"):
    save_dir = Path(model.trainer.save_dir)

if save_dir is not None:
    best_path = save_dir / "weights" / "best.pt"   # YOLO luôn lưu tên "best.pt"
    if best_path.exists():
        WEIGHTS = str(best_path)
        print(f"✅ Fine-tune xong. WEIGHTS mới: {WEIGHTS}")

        try:
            _yolo_model = None
            print("🔁 Đã reset _yolo_model, init_yolo() sẽ load weights mới.")
        except NameError:
            print("ℹ️ _yolo_model chưa được tạo, không cần reset.")
    else:
        print("⚠️ Không tìm thấy best.pt, giữ nguyên WEIGHTS cũ.")
else:
    print("⚠️ Không xác định được save_dir, hãy xem đường dẫn trong runs_finetune/")


🚀 Bắt đầu fine-tune YOLO...
📁 DATA_YAML    : /content/My-Traffic-Train-5/data.yaml
🎯 BASE_WEIGHTS : /content/bestfinal.pt
Ultralytics 8.3.233 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/My-Traffic-Train-5/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=5, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/bestfinal.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=y

In [39]:
# ==================== CELL 4 — RAG LUẬT GIAO THÔNG TỪ PDF ====================

def _pdf_text(path):
    if PdfReader is None:
        return ""
    reader = PdfReader(path)
    txt = []
    for p in reader.pages:
        t = (p.extract_text() or "").replace("\t", " ")
        t = re.sub(r"[ \u00A0]+", " ", t).strip()
        if t:
            txt.append(t)
    return "\n".join(txt)

def _chunk(text, max_tokens=220, overlap=40):
    ws = text.split()
    out = []
    i = 0
    while i < len(ws):
        j = min(i + max_tokens, len(ws))
        out.append(" ".join(ws[i:j]))
        i = j - overlap if j - overlap > i else j
    return out

def build_rag_index_once():
    """Build RAG index 1 lần duy nhất từ PDF luật"""
    chunks_file = f"{RAG_DIR}/chunks.jsonl"
    faiss_file = f"{RAG_DIR}/faiss.index"

    # Nếu đã có index → skip
    if Path(faiss_file).exists() and Path(chunks_file).exists():
        print("✅ RAG index already exists")
        return

    print("🔄 Building RAG index from PDFs...")

    pdf_list = []
    for src in [LAW_PDF_NEW, LAW_PDF_OLD]:
        if src and Path(src).exists():
            pdf_list.append(src)

    if not pdf_list:
        print("⚠️ No PDF found, skip RAG index")
        return

    # Extract chunks
    corpus = []
    for pdf_path in pdf_list:
        raw = _pdf_text(pdf_path)
        parts = re.split(r"\n\s*(Điều\s+\d+[^:\n]*:?)", raw, flags=re.I)

        if len(parts) > 1:
            for head, body in zip(parts[1::2], parts[2::2]):
                for ch in _chunk(head + "\n" + body):
                    corpus.append({
                        "pdf": Path(pdf_path).name,
                        "section": head.strip(),
                        "text": ch
                    })
        else:
            for ch in _chunk(raw):
                corpus.append({
                    "pdf": Path(pdf_path).name,
                    "section": "",
                    "text": ch
                })

    if not corpus:
        print("⚠️ No content extracted from PDFs")
        return

    # Build embeddings
    model = init_embeddings()
    emb = model.encode(
        [c["text"] for c in corpus],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True
    )

    # Save chunks
    with open(chunks_file, "w", encoding="utf-8") as f:
        for c in corpus:
            f.write(json.dumps(c, ensure_ascii=False) + "\n")

    # Save FAISS index
    index = faiss.IndexFlatIP(emb.shape[1])
    index.add(emb)
    faiss.write_index(index, faiss_file)

    print(f"✅ RAG indexed {len(corpus)} chunks → {faiss_file}")

def init_rag():
    """Load RAG index"""
    global _rag_index, _rag_chunks

    chunks_file = f"{RAG_DIR}/chunks.jsonl"
    faiss_file = f"{RAG_DIR}/faiss.index"

    if _rag_index is None and Path(faiss_file).exists():
        print("🔄 Loading RAG index...")
        _rag_index = faiss.read_index(faiss_file)

        with open(chunks_file, "r", encoding="utf-8") as f:
            _rag_chunks = [json.loads(ln) for ln in f]

        print(f"✅ RAG: {_rag_index.ntotal} vectors, {len(_rag_chunks)} chunks")

    return _rag_index, _rag_chunks

def rag_fetch(query: str, k: int = 3) -> List[str]:
    """Search RAG"""
    index, chunks = init_rag()
    if index is None or not chunks:
        return []

    try:
        model = init_embeddings()
        qvec = model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
        D, I = index.search(qvec.astype('float32'), k)

        results = []
        for idx, score in zip(I[0], D[0]):
            if 0 <= idx < len(chunks):
                chunk = chunks[idx]
                text = f"[{chunk.get('section', '')}] {chunk['text'][:150]}"
                results.append(text)
        return results
    except Exception as e:
        print(f"⚠️ RAG error: {e}")
        return []

In [40]:
# ==================== CELL 5: INIT ALL COMPONENTS ====================

def init_all_components():
    """Khởi tạo TẤT CẢ model + RAG index 1 lần."""
    print("\n🔧 Initializing models & indexes...")

    # Models
    init_yolo()
    init_ocr()
    init_embeddings()

    if USE_QWEN:
        init_qwen()

    if USE_CLIP_CONTEXT:
        init_clip()

    # RAG
    build_rag_index_once()
    init_rag()

    print("✅ All components initialized!\n")

# 👉 Chạy luôn khi bạn bấm run cell này
print("🚀 Running init_all_components() ...")
init_all_components()


🚀 Running init_all_components() ...

🔧 Initializing models & indexes...
🔧 Loading YOLO with weights: /content/roadbuddy/runs_finetune/yolo11_roadbuddy_finetune/weights/best.pt (device=0)
🔄 Loading OCR...
🔄 Loading Embeddings: sentence-transformers/all-MiniLM-L6-v2
🔄 Loading Qwen base: Qwen/Qwen2.5-3B-Instruct


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

🔄 Loading QLoRA adapter from: /content/content/qwen2.5-3b-roadbuddy-ft
✅ Qwen + LoRA (RoadBuddy FT) loaded
🔄 Loading CLIP: openai/clip-vit-base-patch32
✅ RAG index already exists
🔄 Loading RAG index...
✅ RAG: 841 vectors, 841 chunks
✅ All components initialized!



#MODULE 1

In [41]:
# ==================== CELL 6B — ĐÁNH GIÁ YOLO SAU FINE-TUNE ====================
from pathlib import Path

print("📊 ĐÁNH GIÁ MODEL YOLO SAU FINE-TUNE")

# Dùng luôn WEIGHTS mới (đã cập nhật ở CELL 6A)
EVAL_WEIGHTS = WEIGHTS
DATA_YAML    = "/content/My-Traffic-Train-5/data.yaml"  # giống DATA_YAML phía trên

print(f"🧠 Dùng weights: {EVAL_WEIGHTS}")
print(f"📁 data.yaml   : {DATA_YAML}")

model_eval = YOLO(EVAL_WEIGHTS)

metrics = model_eval.val(
    data=DATA_YAML,
    imgsz=640,
    batch=8,
    device=0 if torch.cuda.is_available() else "cpu",
    plots=True,        # lưu confusion matrix, PR, F1...
    save_json=True     # nếu cần COCO json
)

print("✅ Đánh giá xong.")
print("🔢 mAP50:", float(metrics.box.map50))
print("🔢 mAP50-95:", float(metrics.box.map))
print("🔢 Precision:", float(metrics.box.mp))
print("🔢 Recall:", float(metrics.box.mr))

if hasattr(model_eval, "trainer") and hasattr(model_eval.trainer, "save_dir"):
    print("📂 Kết quả chi tiết (confusion matrix, PR curve, ...) nằm ở:", model_eval.trainer.save_dir)


📊 ĐÁNH GIÁ MODEL YOLO SAU FINE-TUNE
🧠 Dùng weights: /content/roadbuddy/runs_finetune/yolo11_roadbuddy_finetune/weights/best.pt
📁 data.yaml   : /content/My-Traffic-Train-5/data.yaml
Ultralytics 8.3.233 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLO11n summary (fused): 100 layers, 2,726,368 parameters, 0 gradients, 7.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2227.1±948.1 MB/s, size: 437.7 KB)
val: Scanning /content/My-Traffic-Train-5/valid/labels.cache... 383 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 383/383 631.2Kit/s 0.0s
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 149, len(boxes) = 956. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 48/48 5.6it/s 8.6s
                   all   

In [42]:
# ==================== CELL 6 — MODULE 1: XỬ LÝ VIDEO & NHẬN DIỆN (YOLO) ====================

def get_video_info(vid_path: str) -> Tuple[float, float, int]:
    cap = cv2.VideoCapture(str(vid_path))
    if not cap.isOpened():
        return 0.0, 0.0, 0
    fps = cap.get(cv2.CAP_PROP_FPS) or 0.0
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    duration = frames / fps if fps > 0 else 0.0
    cap.release()
    return fps, duration, frames

def sample_timestamps_by_time(duration: float, step: float, limit: int) -> List[float]:
    if duration <= 0:
        return []
    ts = [round(t, 3) for t in np.arange(0.0, duration, step).tolist()]
    return ts[:limit] if limit > 0 else ts

def read_frame_at(vid_path: str, t_sec: float):
    cap = cv2.VideoCapture(str(vid_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 0.0
    if fps <= 0:
        cap.release()
        return None
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(t_sec * fps))
    ok, frame = cap.read()
    cap.release()
    return frame if ok else None

def extract_keyframes(vid_path: str) -> Tuple[List[float], List[np.ndarray]]:
    """Extract keyframes từ video"""
    _, duration, _ = get_video_info(vid_path)
    ts = sample_timestamps_by_time(duration, SECONDS_PER_SAMPLE, MAX_FRAMES_PER_VID)
    imgs = [read_frame_at(vid_path, t) for t in ts]
    return ts, imgs

def run_yolo_on_frames(model, frames: List[np.ndarray]):
    """YOLO detection"""
    results = [None] * len(frames)
    idxs = [i for i, f in enumerate(frames) if f is not None]
    if not idxs:
        return results

    imgs = [frames[i] for i in idxs]
    r = model.predict(source=imgs, conf=CONF_THRES, iou=IOU_THRES, device=DEVICE, verbose=False)

    for j, ridx in enumerate(idxs):
        results[ridx] = r[j]

    return results

def postprocess_results(result, names: Dict[int, str]) -> List[Dict[str, Any]]:
    """Parse YOLO results"""
    if result is None or not hasattr(result, "boxes"):
        return []
    boxes = result.boxes
    if boxes is None:
        return []

    xyxy = boxes.xyxy.cpu().numpy() if hasattr(boxes, "xyxy") else []
    cls  = boxes.cls.cpu().numpy().astype(int) if hasattr(boxes, "cls") else []
    conf = boxes.conf.cpu().numpy() if hasattr(boxes, "conf") else []

    out = []
    for i in range(len(cls)):
        x1, y1, x2, y2 = map(float, xyxy[i])
        c = int(cls[i])

        # Get original English name from YOLO model's names dictionary
        original_name = names.get(c, str(c))

        # Translate to Vietnamese using LABEL_MAP, fallback to original_name if not found
        translated_name = LABEL_MAP.get(original_name, original_name)

        out.append({
            "cls": c,
            "name": translated_name,
            "conf": float(conf[i]),
            "xyxy": [x1, y1, x2, y2]
        })
    return out

#MODULE 2

In [43]:
# ==================== CELL 7 — MODULE 2: PHÂN TÍCH KHÔNG GIAN-THỜI GIAN & MÔ TẢ CẢNH ====================

def extract_spatial_temporal(keyframes: List[Dict], fps: float) -> Dict:
    """Phân tích không gian-thời gian"""
    features = {
        "trajectories": [],
        "spatial_density": 0,
        "avg_movement_speed": 0,
        "object_counts": Counter()
    }

    # Track objects
    tracks = defaultdict(list)  # {name: [(frame_idx, cx, cy, area), ...]}

    for kf_idx, kf in enumerate(keyframes):
        for obj in kf.get("objects", []):
            bbox = obj.get("xyxy", [0, 0, 0, 0])
            if len(bbox) == 4:
                x1, y1, x2, y2 = bbox
                cx = (x1 + x2) / 2
                cy = (y1 + y2) / 2
                area = max(0, (x2 - x1) * (y2 - y1))

                name = obj.get("name", f"obj_{obj.get('cls')}")
                tracks[name].append((kf_idx, cx, cy, area))
                features["object_counts"][name] += 1

    # Calculate movement speed
    all_speeds = []
    for name, track in tracks.items():
        if len(track) >= 2:
            track.sort(key=lambda x: x[0])
            speeds = []

            for i in range(1, len(track)):
                f1, x1, y1, a1 = track[i-1]
                f2, x2, y2, a2 = track[i]

                dt = (f2 - f1) / fps if fps > 0 else 1.0
                if dt > 0:
                    dist = math.sqrt((x2-x1)**2 + (y2-y1)**2)
                    speed = dist / dt
                    speeds.append(speed)

            if speeds:
                features["trajectories"].append({
                    "name": name,
                    "avg_speed": float(np.mean(speeds)),
                    "max_speed": float(np.max(speeds)),
                    "frames": len(track)
                })
                all_speeds.extend(speeds)

    if all_speeds:
        features["avg_movement_speed"] = float(np.mean(all_speeds))

    # Spatial density
    all_positions = []
    for track in tracks.values():
        all_positions.extend([(x, y) for _, x, y, _ in track])

    if len(all_positions) > 1:
        positions_arr = np.array(all_positions)
        features["spatial_density"] = float(np.std(positions_arr))

    return features

def build_scene_description(keyframes: List[Dict], spatial_features: Dict, fps: float) -> str:
    """Tạo mô tả scene"""
    parts = []

    # Object counts
    obj_counts = spatial_features["object_counts"]
    if obj_counts:
        top_objs = [f"{k}×{v}" for k, v in obj_counts.most_common(5)]
        parts.append(f"Phát hiện: {', '.join(top_objs)}")

    # Movement info
    if spatial_features["avg_movement_speed"] > 0:
        parts.append(f"Chuyển động TB: {spatial_features['avg_movement_speed']:.1f}px/s")

    # Spatial density
    density = spatial_features["spatial_density"]
    if density > 100:
        parts.append("Mật độ cao")
    elif density > 50:
        parts.append("Mật độ trung bình")

    return ". ".join(parts) + "." if parts else "Không có thông tin cảnh."

def build_temporal_facts(keyframes: List[Dict], fps: float) -> List[str]:
    """Tạo temporal facts"""
    facts = []

    light_hist = Counter()
    sign_hist = Counter()

    for kf in keyframes:
        for obj in kf.get("objects", []):
            if obj["conf"] < CONF_THRES:
                continue

            name = obj["name"].lower()

            # Traffic lights
            if any(kw in name for kw in ['den', 'light', 'red', 'green', 'yellow']):
                light_hist[name] += 1
            else:
                sign_hist[name] += 1

    if light_hist:
        dominant_light = light_hist.most_common(1)[0][0]
        facts.append(f"Đèn chủ đạo: {dominant_light}")

    if sign_hist:
        top_signs = [k for k, v in sign_hist.most_common(3)]
        facts.append(f"Biển báo: {', '.join(top_signs)}")

    return facts

In [44]:
# ==================== CELL 8 — OCR: ĐỌC CHỮ TỪ VIDEO ====================

def sample_frames_for_ocr(video_path: str, max_frames: int = 6) -> List[np.ndarray]:
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return []

    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if n <= 0:
        cap.release()
        return []

    idxs = np.linspace(0, max(n-1, 1), max_frames, dtype=int)
    frames = []

    for idx in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, fr = cap.read()
        if ok:
            frames.append(fr)

    cap.release()
    return frames

def ocr_video_texts(video_path: str) -> List[str]:
    ocr = init_ocr()
    texts = []

    for fr in sample_frames_for_ocr(video_path, max_frames=6):
        rgb = fr[:,:,::-1]
        res = ocr.readtext(rgb, detail=0)
        for t in res:
            t = re.sub(r"\s+", " ", t).strip()
            if len(t) >= 2 and t not in texts:
                texts.append(t)

    return texts[:20]

In [45]:
# ==================== CELL 9 — CLIP: NGỮ CẢNH HÌNH ẢNH (VLM CONTEXT) ====================

def clip_embed_frames(frames: List[np.ndarray]) -> Optional[np.ndarray]:
    """Tạo CLIP embedding cho frames"""
    if not USE_CLIP_CONTEXT or not _CLIP_AVAILABLE:
        return None

    model, proc = init_clip()
    if model is None:
        return None

    try:
        images = [Image.fromarray(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)) for f in frames if f is not None]
        if not images:
            return None

        inputs = proc(images=images, return_tensors="pt").to(DEVICE)

        with torch.no_grad():
            feats = model.get_image_features(pixel_values=inputs["pixel_values"])
            feats = torch.nn.functional.normalize(feats, dim=-1)
            v = feats.mean(dim=0)
            v = torch.nn.functional.normalize(v, dim=0)

        return v.detach().cpu().numpy()
    except Exception as e:
        print(f"⚠️ CLIP embed error: {e}")
        return None

In [46]:
# ==================== CELL 10 — PIPELINE CHO 1 VIDEO (MODULE 1 + 2 GỘP LẠI) ====================

def process_video_on_demand(video_path: str) -> Dict[str, Any]:
    """
    Pipeline đầy đủ cho 1 video:
    1. Check cache
    2. Extract keyframes (Module 1)
    3. YOLO detection
    4. Spatial-temporal analysis (Module 2)
    5. Scene description
    6. Temporal facts
    7. OCR
    8. CLIP embedding (optional)
    9. Cache kết quả
    """
    video_name = Path(video_path).stem
    cache_file = Path(CACHE_DIR) / f"{video_name}.json"

    # Check cache
    if cache_file.exists():
        try:
            with open(cache_file, "r", encoding="utf-8") as f:
                cached = json.load(f)
                print(f"✅ Cache hit: {video_name}")
                return cached
        except:
            pass

    print(f"🔄 Processing: {video_name}")

    # Step 1: Extract keyframes
    ts, imgs = extract_keyframes(video_path)
    if not imgs:
        return {"error": "Cannot extract keyframes"}

    # Step 2: YOLO detection
    model = init_yolo()
    names = getattr(model.model, "names", {})
    results = run_yolo_on_frames(model, imgs)

    # Build keyframes data
    keyframes = []
    for i, t in enumerate(ts):
        objs = postprocess_results(results[i], names)
        keyframes.append({"t": float(t), "objects": objs})

    # Step 3: Get video info
    fps, duration, total_frames = get_video_info(video_path)

    # Step 4: Spatial-temporal analysis
    spatial_features = extract_spatial_temporal(keyframes, fps)

    # Step 5: Scene description
    scene_desc = build_scene_description(keyframes, spatial_features, fps)

    # Step 6: Temporal facts
    temporal_facts = build_temporal_facts(keyframes, fps)

    # Step 7: OCR
    try:
        ocr_texts = ocr_video_texts(video_path)
    except:
        ocr_texts = []

    # Step 8: CLIP embedding (optional)
    clip_embedding = None
    if USE_CLIP_CONTEXT:
        try:
            valid_imgs = [img for img in imgs if img is not None]
            if valid_imgs:
                clip_embedding = clip_embed_frames(valid_imgs[:8])
        except:
            pass

    # Build result
    result = {
        "video_name": video_name,
        "fps": float(fps),
        "duration": float(duration),
        "total_frames": int(total_frames),
        "keyframes": keyframes,
        "spatial_features": {
            "object_counts": dict(spatial_features["object_counts"]),
            "avg_movement_speed": spatial_features["avg_movement_speed"],
            "spatial_density": spatial_features["spatial_density"],
            "num_trajectories": len(spatial_features["trajectories"])
        },
        "scene_description": scene_desc,
        "temporal_facts": temporal_facts,
        "ocr_texts": ocr_texts,
        "has_clip_embedding": clip_embedding is not None
    }

    # Cache
    with open(cache_file, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)

    # Save CLIP embedding separately if exists
    if clip_embedding is not None:
        emb_file = Path(CACHE_DIR) / f"{video_name}_clip.npy"
        np.save(emb_file, clip_embedding)

    # Cleanup
    del imgs, results
    gc.collect()
    torch.cuda.empty_cache()

    return result

#MODULE 3

In [47]:
# ==================== CELL 11 — MODULE 3: SUY LUẬN (QWEN + LUẬT + QUY TẮC) ====================

def infer_qwen(q: str, options: List[str], scene: str, labels: List[str],
               ocr: List[str], laws: List[str], temporal: List[str]) -> Tuple[str, str]:
    """Suy luận với Qwen"""
    model, tok = init_qwen()

    if model is None or tok is None:
        return rule_based_inference(q, options, scene, labels, ocr, laws, temporal)

    try:
        # Build context
        ctx_parts = []

        if scene:
            ctx_parts.append(f"Cảnh: {scene[:100]}")

        if temporal:
            ctx_parts.append(f"Thời gian: {temporal[0][:50]}")

        if labels:
            ctx_parts.append(f"Nhãn: {', '.join(labels[:3])}")

        if ocr and any(kw in q.lower() for kw in ['tốc độ', 'km/h', 'số', 'biển']):
            ocr_nums = [x for x in ocr if re.search(r'\d', x)]
            if ocr_nums:
                ctx_parts.append(f"OCR: {ocr_nums[0]}")

        law_context_str = ""
        if laws:
            ctx_parts.append(f"Luật: {laws[0][:80]}")
            law_context_str = " | Luật: " + " | ".join(laws)

        context = " | ".join(ctx_parts)[:400]

        # Build prompt
        opt_txt = "\n".join(f"{chr(65+i)}. {opt}" for i, opt in enumerate(options))

        system_msg = "Bạn là trợ lý lái xe. Trả lời ĐÚNG format: chữ cái A/B/C/D."
        user_msg = f"Câu hỏi: {q}\n\nĐáp án:\n{opt_txt}\n\nThông tin: {context}\n\nTrả lời:"

        messages = [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg}
        ]

        prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inp = tok(prompt, return_tensors="pt", truncation=True, max_length=800).to(model.device)
        prompt_len = inp.input_ids.shape[1]

        with torch.no_grad():
            out = model.generate(
                **inp,
                max_new_tokens=25,
                do_sample=True,
                temperature=0.15,
                top_p=0.9,
                pad_token_id=tok.pad_token_id or tok.eos_token_id
            )

        gen_tokens = out[0][prompt_len:]
        response = tok.decode(gen_tokens, skip_special_tokens=True).strip()

        del out, inp, gen_tokens
        torch.cuda.empty_cache()

        # Parse
        match = re.search(r'\b([ABCD])\b', response, re.IGNORECASE)
        if match:
            ans = match.group(1).upper()
            if 0 <= ord(ans) - 65 < len(options):
                return ans, response[:150] + law_context_str

        # Fallback
        return rule_based_inference(q, options, scene, labels, ocr, laws, temporal)

    except Exception as e:
        print(f"❌ Qwen error: {e}")
        return rule_based_inference(q, options, scene, labels, ocr, laws, temporal)


def rule_based_inference(q: str, options: List[str], scene: str, labels: List[str],
                        ocr: List[str], laws: List[str], temporal: List[str]) -> Tuple[str, str]:
    """Rule-based fallback"""
    import random

    q_lower = q.lower()

    law_context_str = ""
    if laws:
        law_context_str = " | Luật: " + " | ".join(laws)

    # Rule 1: Traffic lights
    if any(kw in q_lower for kw in ['đèn đỏ', 'đèn xanh', 'đèn vàng']):
        for i, opt in enumerate(options):
            opt_lower = opt.lower()
            if 'đèn đỏ' in q_lower and 'dừng' in opt_lower:
                return chr(65 + i), f"Rule: Đèn đỏ phải dừng" + law_context_str
            if 'đèn xanh' in q_lower and 'đi' in opt_lower:
                return chr(65 + i), f"Rule: Đèn xanh được đi" + law_context_str
            if 'đèn vàng' in q_lower and ('chậm' in opt_lower or 'giảm' in opt_lower):
                return chr(65 + i), f"Rule: Đèn vàng giảm tốc" + law_context_str

    # Rule 2: Speed limits from OCR
    if any(kw in q_lower for kw in ['tốc độ', 'km/h', 'giới hạn']):
        numbers = []
        for txt in ocr:
            match = re.findall(r'\d+', txt)
            numbers.extend([int(x) for x in match if 0 < int(x) <= 120])

        if numbers:
            speed_limit = max(numbers)
            for i, opt in enumerate(options):
                if str(speed_limit) in opt:
                    return chr(65 + i), f"Rule: OCR speed={speed_limit}" + law_context_str

    # Rule 3: Stop/Yield signs
    if any(kw in q_lower for kw in ['dừng', 'stop', 'nhường đường']):
        for i, opt in enumerate(options):
            opt_lower = opt.lower()
            if any(kw in opt_lower for kw in ['dừng', 'stop', 'nhường']):
                return chr(65 + i), f"Rule: Sign detected" + law_context_str

    # Rule 4: Check labels
    if labels:
        label_str = ' '.join(labels).lower()
        for i, opt in enumerate(options):
            opt_lower = opt.lower()
            # Match keywords from labels to options
            if any(word in label_str for word in opt_lower.split() if len(word) > 3):
                return chr(65 + i), f"Rule: Label match" + law_context_str

    # Rule 5: Law context
    if laws:
        law_text = ' '.join(laws).lower()
        best_match = -1
        best_score = 0

        for i, opt in enumerate(options):
            opt_words = set(opt.lower().split())
            score = sum(1 for word in opt_words if word in law_text and len(word) > 3)
            if score > best_score:
                best_score = score
                best_match = i

        if best_match >= 0:
            return chr(65 + best_match), f"Rule: Law context" + law_context_str

    # Rule 6: Longest/most detailed option (heuristic)
    longest_idx = max(range(len(options)), key=lambda i: len(options[i]))
    return chr(65 + longest_idx), "Rule: Longest option" + law_context_str


In [49]:
# ==================== CELL 11 — MODULE 3: KẾT HỢP VIDEO + LUẬT → TRẢ LỜI 1 CÂU HỎI ====================

def answer_question(q_data: Dict) -> Tuple[str, str]:
    """
    Trả lời câu hỏi dựa trên video
    Returns: (answer_letter, reasoning)
    """
    # Add a type check for q_data to prevent AttributeError if it's a string
    if not isinstance(q_data, dict):
        return "A", "Error: Input question data is not a dictionary"

    q_id = q_data.get("id", "")
    question = q_data.get("question", "")
    options = q_data.get("options", [])
    video_name = q_data.get("video", "")

    if not options or not video_name:
        return "A", "Missing data"

    # Find video
    video_path = None
    for ext in ['.mp4', '.avi', '.mov']:
        p = Path(VIDEO_ROOT) / f"{video_name}{ext}"
        if p.exists():
            video_path = str(p)
            break

    if not video_path:
        return "A", f"Video not found: {video_name}"

    # Process video
    video_data = process_video_on_demand(video_path)

    if "error" in video_data:
        return "A", video_data["error"]

    # Extract features
    scene = video_data.get("scene_description", "")
    temporal = video_data.get("temporal_facts", [])
    ocr = video_data.get("ocr_texts", [])

    # Get labels from keyframes
    labels = []
    for kf in video_data.get("keyframes", []):
        for obj in kf.get("objects", []):
            name = obj.get("name", "")
            if name and name not in labels:
                labels.append(name)

    # RAG: Fetch relevant laws
    laws = rag_fetch(question, k=2)

    # Inference
    if USE_QWEN:
        answer, reasoning = infer_qwen(question, options, scene, labels, ocr, laws, temporal)
    else:
        answer, reasoning = rule_based_inference(question, options, scene, labels, ocr, laws, temporal)

    return answer, reasoning


In [50]:
# ==================== CELL 12 — CHẠY TOÀN BỘ DATASET & GHI CSV ====================

def process_all_questions():
    """Process tất cả câu hỏi và lưu kết quả"""

    # Load questions
    if not Path(QUESTIONS_JSON).exists():
        print(f"❌ Questions file not found: {QUESTIONS_JSON}")
        return

    with open(QUESTIONS_JSON, "r", encoding="utf-8") as f:
        raw_json_data = json.load(f)
        questions_list = raw_json_data.get("data", [])

    print(f"📋 Loaded {len(questions_list)} questions")

    # Open CSV files for incremental saving
    short_file_path = Path(OUT_SHORT)
    full_file_path = Path(OUT_FULL)

    short_file_exists = short_file_path.exists() and short_file_path.stat().st_size > 0
    full_file_exists = full_file_path.exists() and full_file_path.stat().st_size > 0

    short_f = open(short_file_path, "a", encoding="utf-8", newline="")
    full_f = open(full_file_path, "a", encoding="utf-8", newline="")

    short_writer = csv.writer(short_f)
    full_writer = csv.writer(full_f)

    if not short_file_exists:
        short_writer.writerow(["id", "answer"])
    if not full_file_exists:
        full_writer.writerow(["id", "answer", "reasoning"])

    # Process questions
    start_time = time.time()

    for idx, raw_q_data in enumerate(questions_list, 1):
        t0 = time.time()  # 🔹 bắt đầu đo thời gian cho câu này

        # Ensure raw_q_data is a dictionary before trying to access its keys
        if not isinstance(raw_q_data, dict):
            print(f"  ❌ Error: Item at index {idx-1} in questions list is not a dictionary. Skipping.")
            q_id = f"q{idx}"
            answer = "A"
            reasoning = (
                f"Input question data at index {idx-1} is not a dictionary. "
                f"Expected dict, got {type(raw_q_data).__name__}."
            )
        else:
            q_id = raw_q_data.get("id", f"q{idx}")

            q_data = {
                "id": q_id,
                "question": raw_q_data.get("question"),
                "options": raw_q_data.get("choices", []),
                "video": Path(raw_q_data.get("video_path", "")).stem
            }

            try:
                print(f"\n[{idx}/{len(questions_list)}] Processing: {q_id}")
                answer, reasoning = answer_question(q_data)
                print(f"  ✅ Answer: {answer}")

            except Exception as e:
                print(f"  ❌ Error: {e}")
                answer = "A"
                reasoning = f"Error: {str(e)[:100]}"

        # 🔹 tính thời gian cho câu hiện tại
        dt = time.time() - t0
        print(f"  ⏱ Thời gian xử lý câu {q_id}: {dt:.2f}s")

        # Write results incrementally
        short_writer.writerow([q_id, answer])
        full_writer.writerow([q_id, answer, reasoning])
        short_f.flush()
        full_f.flush()

        # Progress update theo block 10 câu
        if idx % 10 == 0:
            elapsed = time.time() - start_time
            avg_time = elapsed / idx
            remaining = avg_time * (len(questions_list) - idx)
            print(
                f"\n⏱️  Progress: {idx}/{len(questions_list)} | "
                f"Avg: {avg_time:.1f}s/q | ETA: {remaining/60:.1f}min"
            )

    # Close files
    short_f.close()
    full_f.close()

    total_time = time.time() - start_time
    print(f"\n✅ DONE!")
    print(f"   Total time: {total_time/60:.1f} minutes")
    print(f"   Avg per question: {total_time/len(questions_list):.1f}s")
    print(f"   Results saved:")
    print(f"   - {OUT_SHORT}")
    print(f"   - {OUT_FULL}")


#MAIN

In [51]:
# ==================== CELL 13 — MAIN: CHẠY ROADBUDDY PIPELINE ====================

if __name__ == "__main__":
    print("="*60)
    print("🚗 RoadBuddy v3 - Traffic Law QA System")
    print("="*60)

    # Nếu muốn chắc chắn (an toàn) có thể gọi lại, sẽ rất nhanh vì model đã init rồi:
    # init_all_components()

    # Run pipeline
    process_all_questions()

    # Cleanup
    print("\n🧹 Cleaning up...")
    gc.collect()
    torch.cuda.empty_cache()

    print("\n✨ All done!")


🚗 RoadBuddy v3 - Traffic Law QA System
📋 Loaded 405 questions

[1/405] Processing: testa_0001
✅ Cache hit: efc9909e_908_clip_001_0000_0009_Y
  ✅ Answer: A
  ⏱ Thời gian xử lý câu testa_0001: 1.08s

[2/405] Processing: testa_0002
✅ Cache hit: efc9909e_908_clip_001_0000_0009_Y
  ✅ Answer: B
  ⏱ Thời gian xử lý câu testa_0002: 0.67s

[3/405] Processing: testa_0003
✅ Cache hit: 7194eae2_908_clip_004_0020_0028_Y
  ✅ Answer: B
  ⏱ Thời gian xử lý câu testa_0003: 0.44s

[4/405] Processing: testa_0004
✅ Cache hit: 37f8a88b_884_clip_001_0000_0010_Y
  ✅ Answer: B
  ⏱ Thời gian xử lý câu testa_0004: 0.41s

[5/405] Processing: testa_0005
✅ Cache hit: 37f8a88b_884_clip_001_0000_0010_Y
  ✅ Answer: C
  ⏱ Thời gian xử lý câu testa_0005: 0.40s

[6/405] Processing: testa_0006
✅ Cache hit: fb16e5db_884_clip_003_0020_0025_Y
  ✅ Answer: B
  ⏱ Thời gian xử lý câu testa_0006: 0.43s

[7/405] Processing: testa_0007
✅ Cache hit: fb16e5db_884_clip_003_0020_0025_Y
  ✅ Answer: A
  ⏱ Thời gian xử lý câu testa_0007:

KeyboardInterrupt: 

#SO SÁNH ĐÁP ÁN

In [52]:
import io
import pandas as pd
from google.colab import files

# ==== FILE ĐÁP ÁN CHUẨN CỦA BTC (GROUND TRUTH) ====
GT_CSV_PATH = "/content/dapan.csv"   # sửa đúng path file đáp án BTC

# ---- GROUND TRUTH (BTC) ----
gt_df = pd.read_csv(GT_CSV_PATH)
gt_df.columns = [c.strip().lower() for c in gt_df.columns]

if "id" not in gt_df.columns or "answer" not in gt_df.columns:
    if gt_df.shape[1] == 1:
        col = gt_df.columns[0]
        tmp = gt_df[col].astype(str).str.split(":", n=1, expand=True)
        tmp.columns = ["id", "answer"]
        gt_df = tmp
    else:
        raise ValueError("❌ Không tìm thấy cột 'id' và 'answer' trong file ground truth (BTC).")

gt_df = gt_df[["id", "answer"]].copy()
gt_df.rename(columns={"answer": "answer_gt"}, inplace=True)
print(f"📊 Ground truth (BTC) có {len(gt_df)} câu hỏi.\n")

# ---- PREDICTIONS (FILE KẾT QUẢ CỦA BẠN) ----
print("👉 Chọn file CSV KẾT QUẢ của BẠN (id,answer) từ máy...")
uploaded = files.upload()
if not uploaded:
    raise RuntimeError("❌ Không có file nào được chọn!")

pred_name = list(uploaded.keys())[0]
print(f"✅ Đã nhận file: {pred_name}")

pred_df = pd.read_csv(io.BytesIO(uploaded[pred_name]))
pred_df.columns = [c.strip().lower() for c in pred_df.columns]

if "id" not in pred_df.columns or "answer" not in pred_df.columns:
    if pred_df.shape[1] == 1:
        col = pred_df.columns[0]
        tmp = pred_df[col].astype(str).str.split(":", n=1, expand=True)
        tmp.columns = ["id", "answer"]
        pred_df = tmp
    else:
        raise ValueError("❌ Không tìm thấy cột 'id' và 'answer' trong file predictions (của bạn).")

pred_df = pred_df[["id", "answer"]].copy()
pred_df.rename(columns={"answer": "answer_pred"}, inplace=True)
print(f"📊 Predictions (của bạn) có {len(pred_df)} câu hỏi.\n")

# ---- SO SÁNH CHỈ TRÊN NHỮNG CÂU BẠN NỘP ----
# merge theo ID trong file của BẠN (left join từ pred -> gt)
merged = pred_df.merge(gt_df, on="id", how="left")

# nếu có id không tìm thấy trong BTC thì báo nhẹ
missing_gt = merged["answer_gt"].isna().sum()
if missing_gt > 0:
    print(f"⚠️ Có {missing_gt} câu trong file của bạn không tìm thấy đáp án BTC (answer_gt = NaN).")

# chỉ tính trên những câu có đáp án BTC
valid = merged[merged["answer_gt"].notna()].copy()

total = len(valid)          # số câu thực sự so sánh
correct = (
    valid["answer_gt"].astype(str).str.strip().str.upper()
    ==
    valid["answer_pred"].astype(str).str.strip().str.upper()
).sum()
accuracy = correct / total * 100 if total > 0 else 0.0

print("===== KẾT QUẢ SO SÁNH =====")
print(f"🔢 Số câu bạn nộp (có đáp án BTC): {total}")
print(f"✅ Số câu đúng: {correct}")
print(f"❌ Số câu sai: {total - correct}")
print(f"🎯 Tỉ lệ chính xác: {accuracy:.2f}%")

wrong = valid[
    valid["answer_gt"].astype(str).str.strip().str.upper()
    !=
    valid["answer_pred"].astype(str).str.strip().str.upper()
]

if not wrong.empty:
    print("\n🔍 Các câu sai:")
    display(wrong)
else:
    print("\n🎉 Tất cả các câu bạn nộp đều đúng so với đáp án BTC!")

'''
answer_gt = đáp án của BTC (ground truth)
→ đọc từ file cố định: GT_CSV_PATH = "/content/dapan.csv"

answer_pred = đáp án do model của BẠN dự đoán
→ đọc từ file bạn upload lên (answers_short.csv bạn vừa chạy ra).
'''


📊 Ground truth (BTC) có 405 câu hỏi.

👉 Chọn file CSV KẾT QUẢ của BẠN (id,answer) từ máy...


Saving answers_short (3).csv to answers_short (3).csv
✅ Đã nhận file: answers_short (3).csv
📊 Predictions (của bạn) có 202 câu hỏi.

===== KẾT QUẢ SO SÁNH =====
🔢 Số câu bạn nộp (có đáp án BTC): 202
✅ Số câu đúng: 103
❌ Số câu sai: 99
🎯 Tỉ lệ chính xác: 50.99%

🔍 Các câu sai:


,id,answer_pred,answer_gt
0,testa_0001,A,C
2,testa_0003,B,A
3,testa_0004,B,A
5,testa_0006,B,A
6,testa_0007,A,B
...,...,...,...
185,testa_0186,A,B
189,testa_0190,A,B
193,testa_0194,B,A
196,testa_0197,B,A


'\nanswer_gt = đáp án của BTC (ground truth)\n→ đọc từ file cố định: GT_CSV_PATH = "/content/dapan.csv"\n\nanswer_pred = đáp án do model của BẠN dự đoán\n→ đọc từ file bạn upload lên (answers_short.csv bạn vừa chạy ra).\n'

#FINE TUNNING QWEN

In [24]:
# ==================== BUILD QWEN FINETUNE DATASET ====================
import json, csv
from pathlib import Path

# 🚩 SỬA LẠI 3 ĐƯỜNG DẪN NÀY CHO ĐÚNG VỚI BẠN
QUESTIONS_JSON = "/content/public_test.json"          # file câu hỏi
ANS_CSV        = "/content/dapan.csv"          # file id,answer
OUT_FT_JSONL   = "/content/qwen_finetune.jsonl"       # file output để fine-tune

# 1) Load bộ đáp án đúng từ CSV
gt = {}  # id -> "A"/"B"/"C"/"D"

if not Path(ANS_CSV).exists():
    raise FileNotFoundError(f"Không tìm thấy file đáp án: {ANS_CSV}")

with open(ANS_CSV, "r", encoding="utf-8-sig") as f:  # utf-8-sig để xử lý BOM \ufeff
    reader = csv.DictReader(f)

    # Chuẩn hoá tên cột: bỏ khoảng trắng hai bên và chuyển về lowercase
    fieldnames_norm = [name.strip().lower() for name in reader.fieldnames]
    reader.fieldnames = fieldnames_norm

    # Kiểm tra xem có cột id / answer không
    if "id" not in reader.fieldnames or "answer" not in reader.fieldnames:
        raise ValueError(f"Không tìm thấy cột 'id' hoặc 'answer' trong CSV. "
                         f"Header thực tế: {reader.fieldnames}")

    for row in reader:
        qid = row["id"].strip()
        ans = row["answer"].strip().upper()
        if not qid:
            continue
        gt[qid] = ans

print(f"✅ Loaded {len(gt)} ground-truth answers từ {ANS_CSV}")

# 2) Load câu hỏi từ JSON
if not Path(QUESTIONS_JSON).exists():
    raise FileNotFoundError(f"Không tìm thấy file câu hỏi: {QUESTIONS_JSON}")

with open(QUESTIONS_JSON, "r", encoding="utf-8") as f:
    raw = json.load(f)
    questions_list = raw.get("data", [])

print(f"📋 Loaded {len(questions_list)} questions từ {QUESTIONS_JSON}")

# 3) Hàm build prompt giống infer_qwen (bản đơn giản: chỉ question + options)
def build_training_prompt(item):
    q = item.get("question", "")
    options = item.get("choices", [])
    opt_txt = "\n".join(f"{chr(65+i)}. {opt}" for i, opt in enumerate(options))
    user_msg = f"Câu hỏi: {q}\n\nĐáp án:\n{opt_txt}\n\nTrả lời:"
    return user_msg

# 4) Gộp lại thành JSONL cho Qwen fine-tune
n_written = 0
out_path = Path(OUT_FT_JSONL)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w", encoding="utf-8") as f_out:
    for it in questions_list:
        qid = it.get("id")
        if qid not in gt:
            continue  # bỏ qua câu không có đáp án đúng

        ans = gt[qid]
        user_msg = build_training_prompt(it)

        rec = {
            "messages": [
                {
                    "role": "system",
                    "content": "Bạn là trợ lý lái xe. Trả lời ĐÚNG format: chữ cái A/B/C/D."
                },
                {
                    "role": "user",
                    "content": user_msg
                },
                {
                    "role": "assistant",
                    "content": ans
                }
            ]
        }
        f_out.write(json.dumps(rec, ensure_ascii=False) + "\n")
        n_written += 1

print(f"✅ Wrote {n_written} training samples vào {OUT_FT_JSONL}")


✅ Loaded 405 ground-truth answers từ /content/dapan.csv
📋 Loaded 405 questions từ /content/public_test.json
✅ Wrote 405 training samples vào /content/qwen_finetune.jsonl


In [25]:
!pip install -q "transformers>=4.41.0" "accelerate>=0.25.0" bitsandbytes peft datasets


In [26]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from transformers import TrainingArguments, Trainer

BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
DATA_PATH  = "/content/qwen_finetune.jsonl"
OUTPUT_DIR = "/content/qwen2.5-3b-roadbuddy-ft"

# 1) Load dataset JSONL
dataset = load_dataset("json", data_files=DATA_PATH, split="train")

# 2) Tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# 3) QLoRA config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# 4) PEFT LoRA config
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],  # thường dùng cho các LLM transformer
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# 5) Hàm tokenize
def format_example(example):
    # input: messages (system, user, assistant)
    messages = example["messages"]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    example["text"] = prompt
    return example

dataset = dataset.map(format_example)

def tokenize_function(example):
    out = tokenizer(
        example["text"],
        max_length=512,
        truncation=True,
        padding="max_length"
    )
    out["labels"] = out["input_ids"].copy()
    return out

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=dataset.column_names)

# 6) TrainingArguments
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=20,
    save_strategy="epoch",
    bf16=torch.cuda.is_available(),
    report_to="none"
)

# 7) Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

trainer.train()

# 8) Save adapter (LoRA weights)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("✅ Fine-tune xong, model lưu tại:", OUTPUT_DIR)


Generating train split: 0 examples [00:00, ? examples/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# ĐỔI đường dẫn nếu bạn đặt OUTPUT_DIR khác
MODEL_DIR = "/content/qwen2.5-3b-roadbuddy-ft"
ZIP_PATH  = "/content/qwen2.5-3b-roadbuddy-ft.zip"

!zip -r "$ZIP_PATH" "$MODEL_DIR"

from google.colab import files
files.download(ZIP_PATH)


  adding: content/qwen2.5-3b-roadbuddy-ft/ (stored 0%)
  adding: content/qwen2.5-3b-roadbuddy-ft/tokenizer_config.json (deflated 89%)
  adding: content/qwen2.5-3b-roadbuddy-ft/chat_template.jinja (deflated 71%)
  adding: content/qwen2.5-3b-roadbuddy-ft/checkpoint-78/ (stored 0%)
  adding: content/qwen2.5-3b-roadbuddy-ft/checkpoint-78/training_args.bin (deflated 53%)
  adding: content/qwen2.5-3b-roadbuddy-ft/checkpoint-78/scheduler.pt (deflated 62%)
  adding: content/qwen2.5-3b-roadbuddy-ft/checkpoint-78/trainer_state.json (deflated 59%)
  adding: content/qwen2.5-3b-roadbuddy-ft/checkpoint-78/adapter_config.json (deflated 57%)
  adding: content/qwen2.5-3b-roadbuddy-ft/checkpoint-78/rng_state.pth (deflated 26%)
  adding: content/qwen2.5-3b-roadbuddy-ft/checkpoint-78/adapter_model.safetensors (deflated 8%)
  adding: content/qwen2.5-3b-roadbuddy-ft/checkpoint-78/optimizer.pt (deflated 8%)
  adding: content/qwen2.5-3b-roadbuddy-ft/checkpoint-78/README.md (deflated 65%)
  adding: content/qwe

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>